# 4.7. Environment and Distribution Shift

전에는 학습 데이터와 실제 사용 환경의 데이터가 비슷한 분포에서 나올것이다 라는 가정으로 진행했다.

하지만 현실은 다르다.

예를 들어서 실제 고양이, 강아지 사진을 사용했는데, 실제 서비스에서 만화 그림이 입력될 수 있다.

모델 자체에는 문제가 없더라도 학습할 때 본 데이터와 실제 데이터가 다르면 성능이 크게 떨어질 수 있다.

이처럼 학습 데이터와 실제 데이터의 분포가 달라지는 현상을 distrbution shift라고 한다. 분포 변화

## 0. 기본 설정

PyTorch를 불러오고 현재 환경을 확인

In [1]:
%matplotlib inline

import matplotlib.pyplot as plt 
import torch 
from torch import nn 
from torch.utils.data import DataLoader 
from torchvision import datasets, transforms

torch.manual_seed(42) 

print("PyTorch version:", torch.__version__)

PyTorch version: 2.11.0+cu128


## 1. 환경까지 고려해야 하는 이유

머신러닝 모델은 단순히 예측만 하는 것이 아니라, 실제 의사결정에 사용되는 경우가 많다.

예를 들어 대출 상환 여부를 예측하는 모델이 있다고 하자.

학습 데이터에서 우연히 다음과 같은 관계를 발견했다고 가정한다.

구두를 신은 사람은 대출을 잘 갚았다.
운동화를 신은 사람은 대출을 갚지 못한 경우가 많았다.

모델이 이 관계만 보고 구두를 신은 사람에게만 대출을 승인하기 시작하면 사람들은 곧 구두를 신고 대출을 신청할 것이다.

하지만 신발이 바뀌었다고 실제 상환 능력이 좋아진 것은 아니다.

모델의 결정이 사람들의 행동을 바꾸고, 바뀐 행동이 다시 데이터에 영향을 줄 수 있다.

모델의 예측 -> 현실의 의사결정 -> 사용자의 행동 변화 -> 새로운 데이터 분포 -> 기존 모델의 성능 저하  

따라서 모델을 만들 때는 정확도뿐만 아니라 모델이 배포될 환경도 고려해야 한다.

## 2. 분포란 무엇인가

분포는 데이터에 어떤 값이 얼마나 자주 나타나는지를 의미한다.

예를 들어서 이미지 분류에서 이런 요소들이 분포에 포함된다.

- 어떤 종류의 이미지가 많이 들어오는가
- 고양이와 강아지의 비율은 얼마인가
- 사진의 밝기와 해상도는 어떠한가
- 각 이미지에 어떤 label이 붙는가

학습 데이터의 분포를 source distribution이라고 하고, 실제 사용할 데이터의 분포를 target distribution이라고 하자.

$$
q(\mathbf{x}, y)
$$

는 학습 데이터의 분포이고,

$$
p(\mathbf{x}, y)
$$

는 실제 사용 환경의 분포라고 표현할 수 있다.

두 분포가 아무 규칙 없이 완전히 다르게 바뀐다면 모델이 새로운 환경에서도 잘 작동하도록 만드는 것은 사실상 불가능하다.

따라서 분포가 어떤 방식으로 변했는지에 관한 가정이 필요하다.

## 3. 분포 변화의 세가지 종류

### 3.1 Covariate Shift
Covariate shift는 입력 데이터의 모습이나 비율은 달라졌지만, 입력과 정답 사이의 관계는 그대로인 경우​이다.

$$
q(\mathbf{x}) \neq p(\mathbf{x})
$$

하지만

$$
q(y \mid \mathbf{x}) = p(y \mid \mathbf{x})
$$

이다.
여기서
$$
P(y \mid \mathbf{x})
$$

는 입력 $\mathbf{x}$가 주어졌을 때 정답이 $y$일 확률이다.

예를 들어서 이럴수 있다.

학습 데이터: 실제 고양이와 강아지 사진
실제 데이터: 고양이와 강아지의 만화 그림

입력 이미지의 스타일은 달라졌지만, 고양이는 여전히 고양이고 강아지는 여전히 강아지다.

입력의 분포는 달라졌지만 정답을 결정하는 기준은 바뀌지 않았다.

### 3.2 Label Shift
Label shift는 각 label이 나타나는 비율이 달라진 경우이다.

$$
q(y) \neq p(y)
$$

하지만

$$
q(\mathbf{x} \mid y) = p(\mathbf{x} \mid y)
$$

라고 가정한다.

예를 들어서 질병 진단 모델을 생각해 보자.

학습 데이터에서는 다음과 같을 수 있다.

건강한 사람: 50%
질병이 있는 사람: 50%

하지만 실제 병원에서는 다음과 같을 수 있다.

건강한 사람: 90%
질병이 있는 사람: 10%

질병이 있는 사람에게 나타나는 증상 자체는 비슷하지만 질병의 발생 비율이 달라졌다.

이것이 label shift이다.

### 3.3 Concept Shift
Concept shift는 입력과 정답 사이의 관계 또는 label의 정의 자체가 달라진 경우이다.

$$
q(y \mid \mathbf{x}) \neq p(y \mid \mathbf{x})
$$

예를 들어 시간이 지나면서 다음과 같은 기준이 바뀔 수 있다.

- 정신 질환의 진단 기준
- 유행하는 패션의 기준
- 직업을 구분하는 기준
- 어떤 표현을 특정 단어로 번역하는 방식

동일한 입력이 주어져도 과거와 현재의 정답이 달라질 수 있다는 뜻이다.

Concept shift는 세 가지 분포 변화 중 가장 대응하기 어렵다.

## 4. 실제 발생하는 문제

### 의료 진단
어떤 질병이 주로 고령 남성에게 발생한다고 하자.

질병 환자의 혈액은 병원에서 수집하고, 건강한 사람의 혈액은 대학생에게서 수집했다면 모델은 질병 자체가 아니라 다음 차이를 학습할 수 있다.

- 나이
- 호르몬
- 식습관
- 운동량
- 음주 습관

모델의 정확도가 높더라도 실제로는 질병 유무가 아니라 고령 환자와 대학생의 차이를 구분한 것일 수 있다.

---

### 자율주행

실제 도로 데이터를 수집하기 어렵다는 이유로 게임 엔진에서 만든 도로 이미지만 학습할 수 있다.

이 경우에 모델은 도로의 본질적인 특징이 아니라 게임 엔진에서 반복되는 단순한 texture를 학습할 수 있다.

가상 환경에서는 높은 정확도가 나오지만 실제 도로에서는 실패할 수 있다.

--- 

### 시간에 따라 변하는 분포

분포가 한 번에 크게 바뀌지 않고 천천히 변하는 경우도 있다.

이를 nonstationary distribution이라고 한다.

예시는 다음과 같다.

- 스팸 필터가 기존 스팸은 잡지만 새로운 유형의 스팸은 잡지 못함
- 겨울 상품 추천 모델이 봄에도 크리스마스 상품을 추천함
- 새로운 기기가 출시됐는데 광고 모델이 이를 반영하지 못함
- 뉴스 추천 모델이 오래된 관심사에 계속 의존함

이런 모델은 한 번 학습하고 끝내면 안 되며 지속적으로 새로운 데이터를 반영해야 한다.

## 5. Empirical Risk와 실제 Risk

모델을 학습할 때는 일반적으로 학습 데이터의 평균 loss를 최소화한다.

$$
\frac{1}{n}
\sum_{i=1}^{n}
l(f(\mathbf{x}_i), y_i)
$$

여기서

- $f(\mathbf{x}_i)$는 모델의 예측
- $y_i$는 정답
- $l$은 loss function
- $n$은 학습 데이터 개수

이 값을 empirical risk (경험적 위험)이라고 한다.

현재 가지고 있는 학습 데이터에서 계산한 평균 loss이다.

하지만 우리가 정말 줄이고 싶은 것은 현실의 모든 데이터에서 발생할 평균 loss이다.

$$
E_{p(\mathbf{x},y)}
\left[
l(f(\mathbf{x}),y)
\right]
$$

이를 단순히 risk라고 한다.

현실의 모든 데이터를 수집할 수 없으므로 학습 데이터의 평균 loss인 empirical risk를 이용해 실제 risk를 근사한다.

**Empirical risk**
= 가지고 있는 학습 데이터의 평균 loss

**Risk**
= 현실의 전체 데이터 분포에서 발생할 평균 loss

학습 데이터가 실제 환경을 잘 대표해야 empirical risk를 줄이는 것이 실제 risk를 줄이는 것으로 이어진다.

## 6. Covariate Shift 보정

Covariate shift에서는 학습 데이터와 실제 데이터에서 입력 분포가 다르다.

학습 입력 분포: q(x)
실제 입력 분포: p(x)

이때 실제 환경에서 자주 나타나는 학습 데이터에는 더 큰 중요도를 주고, 실제 환경에서 잘 나타나지 않는 데이터에는 작은 중요도를 줄 수 있다.

각 학습 데이터의 weight를 다음과 같이 정의한다.
$$
\frac{p(\mathbf{x}_i)}
{q(\mathbf{x}_i)}
$$

- 실제 환경에서 자주 나타나면 $\beta_i$가 커진다.
- 학습 데이터에만 자주 나타나면 $\beta_i$가 작아진다.

이 weight를 loss에 곱한다.

$$
\frac{1}{n}
\sum_{i=1}^{n}
\beta_i
l(f(\mathbf{x}_i),y_i)
$$

이를 `weighted empirical risk minimization`이라고 한다.

쉽게 말하면 다음과 같다.

실제 환경과 비슷한 학습 데이터
→ loss를 크게 반영

실제 환경과 동떨어진 학습 데이터
→ loss를 작게 반영

하지만 실제로는 $p(\mathbf{x})$와 $q(\mathbf{x})$를 정확히 알 수 없다.

그래서 다음과 같은 이진 분류기를 따로 학습할 수 있다.

학습 데이터에서 가져온 sample → class 0  
실제 환경에서 가져온 sample → class 1

이 분류기가 두 데이터를 쉽게 구분한다면 두 분포가 상당히 다르다는 뜻이다.

분류 결과를 이용하면 각 학습 sample의 importance weight를 근사할 수 있다.

## 7. Label Shift와 Concept Shift 보정

### Label Shift 보정

Label shift에서는 label의 비율이 변했으므로 label별 weight를 조절한다.
$$
\frac{p(y_i)}
{q(y_i)}
$$

예를 들어 학습 데이터보다 실제 환경에서 특정 class가 두 배 자주 등장한다면 해당 class의 학습 sample에 더 큰 weight를 줄 수 있다.

Target 데이터에는 정답 label이 없기 때문에 target label 분포를 직접 계산하기는 어렵다.

D2L에서는 다음 정보를 활용할 수 있다고 설명한다.

- validation set에서 계산한 confusion matrix
- 실제 target 데이터에서 모델이 예측한 class의 비율

이를 이용해 target 환경의 label 분포를 추정한다.

### Concept Shift 보정

Concept shift는 정답을 결정하는 규칙 자체가 바뀌었기 때문에 단순한 weight 조절만으로 해결하기 어렵다.

변화가 매우 크다면 새로운 label을 수집해 다시 학습해야 한다.

변화가 천천히 진행된다면 기존 모델의 파라미터를 시작점으로 사용하고 새로운 데이터로 조금씩 추가 학습할 수 있다.

기존 모델 -> 새로운 데이터 수집 -> 추가 학습 -> 변화한 환경에 적응

Covariate shift와 label shift의 보정 수식은 고급 내용이다. 현재 단계에서는 데이터마다 중요도를 다르게 주어 실제 환경과 비슷한 분포를 만들려는 방법이라고 이해하면 충분하다.

## 8. 학습 환경의 종류

### Batch Learning
전체 학습 데이터를 한 번에 모아 모델을 학습한 뒤 배포한다.
배포 후에는 모델을 거의 업데이트하지 않는다.

데이터 수집 → 학습 → 배포 → 사용

학습 데이터와 실제 데이터의 분포가 안정적일 때 적합하다.

### Online Learning
데이터가 하나씩 계속 들어오고, 결과가 확인될 때마다 모델을 업데이트한다.

현재 모델 -> 새로운 입력 -> 예측 -> 정답 확인 -> loss 계산 -> 모델 업데이트

주가 예측, 광고 클릭률 예측처럼 환경이 계속 변하는 문제에 사용할 수 있다.

### Bandit
선택 가능한 행동이 제한된 online learning 문제이다.

예를 들어 여러 광고 중 하나를 선택하고, 사용자가 클릭했는지를 관찰할 수 있다.
핵심 문제는 다음 두 가지 사이의 균형이다.

Exploration: 아직 잘 모르는 행동을 시도
Exploitation: 현재 가장 좋아 보이는 행동을 선택

### Control과 Reinforcement Learning

Control에서는 이전 행동이 이후 환경에 영향을 준다.
Reinforcement learning에서는 행동, 보상, 환경 변화가 반복된다.

예를 들어 자율주행차의 행동은 주변 차량의 다음 행동에 영향을 줄 수 있다.

따라서 각 입력을 독립적으로 예측하는 일반적인 supervised learning만으로는 충분하지 않을 수 있다고 한다.

## 9. 공정성, 책임성과 피드백 루프

머신러닝 모델을 실제로 배포하면 단순한 예측기가 아니라 사람에게 영향을 주는 의사결정 시스템이 된다.

따라서 accuracy만 높다고 좋은 시스템이라고 볼 수 없다.

예를 들어서 의료 모델은 전체 accuracy가 높아도 특정 연령이나 집단에서 성능이 매우 낮을 수 있다.

또한 잘못된 예측마다 발생하는 피해가 다를 수 있다.

거짓 양성의 피해 != 거짓 음성의 피해

모델의 예측이 새로운 학습 데이터를 만들어 내는 feedback loop도 주의해야 한다.

예를 들어서 범죄 발생 가능성이 높다고 예측된 지역에 경찰을 더 많이 배치한다고 하자.

특정 지역을 위험하다고 예측 -> 
해당 지역에 경찰을 더 배치 -> 
해당 지역에서 더 많은 범죄가 발견됨 -> 
학습 데이터에 범죄 기록이 더 많이 쌓임 -> 
모델이 해당 지역을 더욱 위험하다고 예측

실제로 범죄가 더 많이 발생해서가 아니라 더 많이 관찰했기 때문에 데이터가 증가했을 가능성이 있다.

이런 순환이 계속되면 모델의 편향이 스스로 강화될 수 있다.

따라서 실제 머신러닝 시스템에서는 다음을 지속적으로 확인해야 한다.

- 어떤 집단에서 모델이 실패하는가
- 학습 데이터가 실제 환경을 대표하는가
- 모델의 결정이 미래 데이터에 영향을 주는가
- accuracy 이외에 어떤 비용과 위험이 존재하는가
- 데이터 분포가 시간에 따라 변하고 있는가

## 10. 오늘의 정리

- 학습 데이터와 실제 데이터가 서로 다른 분포에서 나오는 현상을 distribution shift라고 한다.
- 모델의 test accuracy가 높아도 실제 환경의 데이터 분포가 다르면 실패할 수 있다.
- 모델의 결정이 사용자의 행동과 미래 데이터의 분포를 바꿀 수도 있다.
- Covariate shift는 입력 분포 $P(\mathbf{x})$가 변하는 경우이다.
- Covariate shift에서는 $P(y \mid \mathbf{x})$가 유지된다고 가정한다.
- Label shift는 class 비율 $P(y)$가 변하는 경우이다.
- Label shift에서는 $P(\mathbf{x} \mid y)$가 유지된다고 가정한다.
- Concept shift는 입력과 정답 사이의 관계 $P(y \mid \mathbf{x})$가 변하는 경우이다.
- Empirical risk는 학습 데이터에서 계산한 평균 loss이다.
- Risk는 실제 전체 데이터 분포에서 기대되는 평균 loss이다.
- 학습 데이터가 실제 환경을 대표해야 empirical risk가 실제 risk를 잘 근사한다.
- Covariate shift는 실제 환경과 비슷한 sample의 loss에 더 큰 weight를 주어 보정할 수 있다.
- Concept shift가 크면 새로운 label을 수집하고 모델을 다시 학습해야 할 수 있다.
- 환경이 지속적으로 변한다면 모델도 새로운 데이터로 계속 업데이트해야 한다.
- 실제 머신러닝 시스템은 accuracy뿐만 아니라 공정성, 피해 비용, feedback loop를 함께 고려해야 한다.